In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from math import ceil
import matplotlib.animation as animation
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dim = 2
N = 512
L = 10.0
n_lattice = int(ceil(N**(1/dim)))
spacing = L / n_lattice
indexing = torch.cartesian_prod(*[torch.arange(n_lattice) for _ in range(dim)]).to(device)
indexing = indexing[torch.randperm(indexing.size(0), device=device)[:N]]
x = (indexing + 0.5) * spacing
sigma = ((0.25 * spacing + 0.25 * spacing * torch.rand(N, device=device))**2)[:, None, None] * torch.eye(dim, device=device)[None, :, :]
v0 = (spacing)**dim
Kv = 0.1
Gamma = 10.0
gamma_r = 100.0
D = 0.1
use_self_propulsion = False
if dim  == 2 and use_self_propulsion:
    D_r = 0.1
    vel0 = 0.1
    thetas = 2 * np.pi * torch.rand(N, device=device)

In [ ]:
def compute_quantities(x, sigma, use_pbc, L):
    N = x.size(0)
    sigma_vals_i, sigma_vecs_i = torch.linalg.eigh(sigma)
    det_sigma_i = torch.prod(sigma_vals_i, dim=-1)
    inv_sigma_i = torch.einsum("iac,ic,ibc->iab", sigma_vecs_i, 1.0/sigma_vals_i, sigma_vecs_i)
    inv_sqrt_sigma_i = torch.einsum("iac,ic,ibc->iab", sigma_vecs_i, 1.0/torch.sqrt(sigma_vals_i), sigma_vecs_i)
    sum_sigma_ij = sigma[:, None] + sigma[None, :]
    det_sum_sigma_ij = torch.det(sum_sigma_ij)
    inv_sum_sigma_ij = torch.linalg.inv(sum_sigma_ij)
    r_ij = x[:, None] - x[None, :]
    if use_pbc:
        r_ij = r_ij - L * torch.round(r_ij / L)
    # resulting shape is (N*N, dim, 1):
    sigma_inv_r_ij = torch.bmm(inv_sum_sigma_ij.view(-1, dim, dim), r_ij.view(-1, dim, 1))
    # resulting shape is (N*N, dim, dim):
    outer_sigma_inv_r_ij = torch.bmm(sigma_inv_r_ij, sigma_inv_r_ij.transpose(1, 2))
    sigma_inv_r_ij = sigma_inv_r_ij.view(N, N, dim)
    outer_sigma_inv_r_ij = outer_sigma_inv_r_ij.view(N, N, dim, dim)
    distance_ij = torch.einsum("ija,ija->ij", sigma_inv_r_ij, r_ij)

    w_ij = torch.exp(-0.5 * (distance_ij + torch.log(det_sum_sigma_ij) - torch.log(det_sigma_i[:, None]) - torch.log(det_sigma_i[None, :])) )

    return det_sigma_i, inv_sigma_i, inv_sqrt_sigma_i, r_ij, det_sum_sigma_ij, inv_sum_sigma_ij, sigma_inv_r_ij, distance_ij, w_ij, outer_sigma_inv_r_ij

In [ ]:
use_pbc = True
indices = torch.arange(N, device=device)
eye_mask = 1 - torch.eye(N, device=device)

dt = 1e-5
T_max = 1.0
n_steps = int(T_max / dt)

snapshot_interval = 50
snapshots_x = []
snapshots_sigma = []

sigma_mobility = 0.1

system_size = L  # (you weren’t updating it anyway)

for step in tqdm(range(n_steps)):

    # --- keep sigma symmetric (do less often if stable) ---
    if step % 10 == 0:
        sigma = 0.5 * (sigma + sigma.transpose(-2, -1))

    # --- snapshots (avoid GPU sync cost) ---
    if step % snapshot_interval == 0:
        snapshots_x.append(x.detach().cpu())
        snapshots_sigma.append(sigma.detach().cpu())

    # --- heavy compute ---
    (
        det_sigma_i,
        inv_sigma_i,
        inv_sqrt_sigma_i,
        r_ij,
        det_sum_sigma_ij,
        inv_sum_sigma_ij,
        sigma_inv_r_ij,
        distance_ij,
        w_ij,
        outer_sigma_inv_r_ij,
    ) = compute_quantities(x, sigma, use_pbc, L)

    # --- mask diagonal ONCE ---
    w_ij = w_ij * eye_mask

    # --- scalars ---
    v_i = torch.sqrt(det_sigma_i)

    trace_inv_sigma_i = torch.diagonal(
        inv_sigma_i, dim1=-2, dim2=-1
    ).sum(-1)

    # --- sigma self force ---
    self_force_sigma = (
        Kv * (v0 - v_i)[:, None, None]
        + Gamma * (trace_inv_sigma_i[:, None, None] / 2.0 - inv_sqrt_sigma_i)
    ) * (v_i[:, None, None] * inv_sigma_i)

    # --- overlap term ---
    overlap_grad_term = (
        -inv_sum_sigma_ij
        + inv_sigma_i[:, None]
        + outer_sigma_inv_r_ij
    )

    # --- sigma update (fused, no big temp tensors) ---
    sigma += (0.5 * dt * sigma_mobility) * self_force_sigma

    sigma -= (0.5 * gamma_r * dt * sigma_mobility) * torch.sum(
        overlap_grad_term * w_ij[:, :, None, None],
        dim=1
    )

    # --- position update (fused) ---
    x += (dt * gamma_r) * torch.sum(
        sigma_inv_r_ij * w_ij[:, :, None],
        dim=1
    )

    # --- noise ---
    x += torch.sqrt(torch.tensor(2 * D * dt, device=device)) * torch.randn_like(x)


# --- convert snapshots AFTER loop ---
snapshots_x = torch.stack(snapshots_x).cpu().numpy()
snapshots_sigma = torch.stack(snapshots_sigma).cpu().numpy()

In [ ]:
from matplotlib.patches import Ellipse
from matplotlib.collections import PolyCollection
from matplotlib.transforms import Affine2D
import matplotlib.cm as cm
import matplotlib.colors as mcolors

fig, ax = plt.subplots()

covs_snapshots_vals, covs_snapshots_vecs = np.linalg.eigh(snapshots_sigma)
vols_snapshots = np.sqrt(np.linalg.det(snapshots_sigma))
vmin, vmax = np.nanquantile(vols_snapshots, [0.1, 0.9])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

unit_circle = Ellipse(xy=(0,0), width=2, height=2).get_path().vertices

def make_verts(frame):
    verts = []
    for i in range(N):
        vals = covs_snapshots_vals[frame, i]
        vecs = covs_snapshots_vecs[frame, i]
        a, b = np.sqrt(vals[1]), np.sqrt(vals[0])
        angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
        cx, cy = snapshots_x[frame, i]
        t = Affine2D().scale(a, b).rotate_deg(angle).translate(cx, cy)
        verts.append(t.transform(unit_circle))
    return verts

col = PolyCollection(make_verts(0), cmap=cmap, norm=norm, alpha=0.7)
col.set_array(vols_snapshots[0])
ax.add_collection(col)
plt.colorbar(col, ax=ax)

ax.set_xlim(snapshots_x[..., 0].min(), snapshots_x[..., 0].max())
ax.set_ylim(snapshots_x[..., 1].min(), snapshots_x[..., 1].max())
ax.set_aspect('equal')

def update(frame):
    col.set_verts(make_verts(frame))
    col.set_array(vols_snapshots[frame])
    return col,

ani = animation.FuncAnimation(fig, update, frames=len(snapshots_x), blit=True)
ani.save('generic_evolution.mp4', fps=24)
plt.show()

In [ ]:
fig, ax = plt.subplots()
covs_snapshots_vals, covs_snapshots_vecs = np.linalg.eigh(snapshots_sigma)
vols_snapshots = np.sqrt(np.linalg.det(snapshots_sigma))
vmin, vmax = np.nanquantile(vols_snapshots, [0.1, 0.9])

scat = ax.scatter(
    snapshots_x[0, :, 0], snapshots_x[0, :, 1],
    c=vols_snapshots[0],
    s=(20*np.sqrt(vols_snapshots[0]))**2,
    cmap='viridis', vmin=vmin, vmax=vmax
)
plt.colorbar(scat, ax=ax)
plt.xlim(0, L)
plt.ylim(0, L)
ax.set_aspect('equal')

def update(frame):
    scat.set_offsets(snapshots_x[frame])
    scat.set_sizes((20*np.sqrt(vols_snapshots[frame]))**2)
    scat.set_array(vols_snapshots[frame])   # ← correct way
    return scat,

ani = animation.FuncAnimation(fig, update, frames=len(snapshots_x), blit=True)
ani.save('generic_evolution.mp4', fps=24)
plt.show()

In [ ]:
pos = x.cpu().numpy()
covs = sigma.cpu().numpy()
vols = np.sqrt(np.linalg.det(covs))
fig, ax = plt.subplots()
plt.scatter(pos[:, 0], pos[:, 1], c=vols, s=400*vols)
plt.xlim(0, L)
plt.ylim(0, L)
plt.colorbar()
ax.set_aspect('equal')
plt.show()